# 04 · Retrain and compare: real only vs ordinary augmentation vs oversampling vs GAN images

Phase 4 of the project. The same four models, the same split and the same training settings as notebook 02. **Only
the training images change:**

| Row | Training images | Images per epoch | What it shows |
|---|---|---|---|
| A | Real only (notebook 02, final run) | 12,698 | the baseline |
| B | Real + ordinary augmentation: flips, rotation, zoom, brightness | 12,698 | what the cheap method gives |
| C | Real, small classes repeated until they have 2,000 (oversampling) | 21,747 | whether balancing alone is enough, without a GAN |
| D | Real + the GAN images of notebook 03_gan_generate (small classes filled up to 2,000) | about 21,747 | **the GAN result** |
| E | Real + GAN images + ordinary augmentation | about 21,747 | whether the two add up |

Row A is not trained again: its numbers come from notebook 02's final run, which used exactly these settings.
Rows C and D have the same number of images per class, so the only difference between them is *repeated real
images* versus *new GAN images*. Validation and test use real images only, and every model is scored on the test
split once, at the end of its run, as in notebook 02.

**This notebook continues where it stopped.** Rows B–E are 4 rows × 4 models × 3 seeds = 48 runs, estimated at 20–25
GPU-session hours across 2–3 commits, based on notebook 02 timings. Actual runtime is unmeasured for this notebook;
if all runs use all 30 epochs, the same speed estimate is about 33 hours. Each commit trains as many missing runs as
fit into `TIME_LIMIT_HOURS` (11), saves them, and draws the comparison with everything finished so far. Then commit
again with the notebook's last version attached as input, until it prints *All 48 runs are done*. Check available
quota before each commit and leave about one hour beyond `TIME_LIMIT_HOURS` (for example, 7.0 with 8 hours left).

| Row | GPU time (estimate, T4) |
|---|---|
| B | 3–4 hours |
| C | 5–6 hours |
| D | 5–6 hours |
| E | 6–7 hours |

**How to run it on Kaggle**

- **Add Input** → `plantvillage-dataset` (by *abdallahalidev*).
- **Add Input** → **Your Work** → **Notebooks** → notebook 01 (`splits.csv`), notebook 02 (the final run, for row A),
  and notebook 03_gan_generate (for `synthetic/`; needed for rows D and E only. Without it, rows B and C are
  trained and D and E wait).
- From the second run on: **Add Input** → **Your Work** → **Notebooks** → *this* notebook (its latest version). The
  notebook finds `04_runs.json` under `/kaggle/input` and continues from it. If Kaggle does not list the notebook
  itself, download `results/04_runs.json` and `models/` from the Output tab and add them with **Add Input → Upload**.
- **Settings**: Accelerator **GPU T4 x2** (one GPU is used, as in notebook 02), **Internet on** (ImageNet weights).
- **Save Version → Save & Run All (Commit)**.
- Optional first: set `SMOKE_TEST = True` for a 10-minute run (MobileNetV2, 1 seed, 2 epochs, no test) that checks
  everything works; its files are named `04_smoke_*` and are never used as results.

In [ ]:
import os, json, math, time, random, shutil, hashlib, fnmatch, warnings
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, PowerNorm, TwoSlopeNorm
from PIL import Image
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
import torch
from torch import nn
from torchvision import models

NOTEBOOK_START = time.time()

ROWS = ['B', 'C', 'D', 'E']   # the rows trained by this notebook; finished runs are skipped
SMOKE_TEST = False            # True: a short test of the whole notebook (MobileNetV2, 1 seed, 2 epochs, no test split)
TIME_LIMIT_HOURS = 11.0       # start no run that might not finish within this time (Kaggle stops a run at 12 hours)
FILL_TARGET = 2000            # rows C, D, E: classes with fewer training images are filled up to this many
# Ordinary augmentation (rows B and E): new random values for every image, every time it is used
AUG = {'flip': True, 'rotate_degrees': 20, 'zoom': 1.2, 'brightness': 0.2}
ROW_NAMES = {'A': 'Real only', 'B': 'Real + ordinary augmentation', 'C': 'Real, small classes repeated',
             'D': 'Real + GAN', 'E': 'Real + GAN + ordinary augmentation'}
ALL_ROWS = ['A', 'B', 'C', 'D', 'E']

SEEDS = [0, 1, 2]
MODELS = ['mobilenet_v2', 'resnet50', 'custom_cnn', 'vgg16']   # the fast ones first
NAMES = {'custom_cnn': 'Custom CNN', 'vgg16': 'VGG16', 'resnet50': 'ResNet50', 'mobilenet_v2': 'MobileNetV2'}

# Training settings: exactly those of notebook 02 (checked against its saved settings below)
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 30
PATIENCE = 5
LR = {'custom_cnn': 1e-3, 'vgg16': 1e-4, 'resnet50': 1e-4, 'mobilenet_v2': 1e-4}
WEIGHT_DECAY = 0.01
PRETRAINED = True
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
EVAL_BATCH = 128
SPLIT_FINGERPRINT = '7cf95c207559517a'

if SMOKE_TEST:
    SEEDS, MODELS, EPOCHS = [0], ['mobilenet_v2'], 2
PREFIX = '04_smoke_' if SMOKE_TEST else '04_'   # all output files start with this
EVAL = 'val' if SMOKE_TEST else 'test'

INPUT_ROOT = Path(os.environ.get('INPUT_ROOT', '/kaggle/input'))
OUT_DIR = Path(os.environ.get('OUT_DIR', '/kaggle/working'))
RESULTS_OUT, MODELS_OUT = OUT_DIR / 'results', OUT_DIR / 'models'
RESULTS_OUT.mkdir(parents=True, exist_ok=True)
MODELS_OUT.mkdir(parents=True, exist_ok=True)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    try:
        nn.functional.conv2d(torch.ones(1, 1, 4, 4, device=DEVICE), torch.ones(1, 1, 3, 3, device=DEVICE))
    except RuntimeError as e:
        raise RuntimeError(f'The GPU {torch.cuda.get_device_name()} does not work with this PyTorch ({e}). '
                           'Choose Settings → Accelerator → GPU T4 x2 and run again.') from None
    torch.backends.cudnn.benchmark = True
    print('GPU:', torch.cuda.get_device_name(), '| PyTorch', torch.__version__)
elif Path('/kaggle').exists():
    raise RuntimeError('No GPU. Choose Settings → Accelerator → GPU T4 x2 and run again.')
else:
    DEVICE = torch.device('cpu')
    print('No GPU: running on the CPU (only for a short test run on another computer)')
AMP = DEVICE.type == 'cuda'
warnings.filterwarnings('ignore', message='Detected call of `lr_scheduler')
print(f'Rows {ROWS}, seeds {SEEDS}' + (' — SMOKE TEST' if SMOKE_TEST else ''))

# Chart style of the project (as in notebooks 01–03)
SURFACE, INK, INK_2, GRID, BLUE, ORANGE = '#fcfcfb', '#0b0b0b', '#52514e', '#e4e3df', '#2a78d6', '#eb6834'
BLUES = LinearSegmentedColormap.from_list('blues', [SURFACE, '#cde2fb', '#86b6ef', '#3987e5', '#1c5cab', '#0d366b'])
# diverging: red = lower than row A, grey = the same, blue = higher
DIVERGING = LinearSegmentedColormap.from_list('diverging', ['#a32d2c', '#e34948', '#f0efec', '#3987e5', '#1c5cab'])
plt.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
    'text.color': INK, 'axes.labelcolor': INK_2, 'xtick.color': INK_2, 'ytick.color': INK_2,
    'axes.edgecolor': GRID, 'axes.linewidth': 1, 'grid.color': GRID, 'grid.linewidth': 1,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 11, 'axes.titlesize': 13, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'savefig.dpi': 150, 'savefig.bbox': 'tight',
})

## 1. The inputs

The split (notebook 01), row A (notebook 02's `02_final_runs.json`), the GAN images (notebook 03_gan_generate's
`synthetic/synthetic.csv`) and the runs this notebook has already finished (`04_runs.json`, from its last version).

In [ ]:
def find_files(root, pattern):
    """Files under root whose name matches pattern. Class folders (names with ___) only hold images, so they are skipped."""
    found = []
    if not Path(root).exists():
        return found
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames.sort()
        dirnames[:] = [d for d in dirnames if '___' not in d]
        found += [Path(dirpath) / f for f in sorted(filenames) if fnmatch.fnmatch(f, pattern)]
    return found

def find_color_dir(root):
    for dirpath, dirnames, _ in os.walk(root):
        dirnames.sort()
        if Path(dirpath).name.lower() == 'color' and any(d.startswith('Tomato') for d in dirnames):
            return Path(dirpath)
        dirnames[:] = [d for d in dirnames if '___' not in d]
    raise FileNotFoundError(f'No PlantVillage "color" folder under {root}. '
                            'Add the Kaggle dataset abdallahalidev/plantvillage-dataset as input.')

def fingerprint(splits):
    return hashlib.sha256('\n'.join(sorted(splits['path'] + ',' + splits['split'])).encode()).hexdigest()[:16]

found = {f: pd.read_csv(f) for f in find_files(INPUT_ROOT, 'splits.csv')}
final_split = [f for f, s in found.items() if fingerprint(s) == SPLIT_FINGERPRINT]
if not final_split:
    raise FileNotFoundError('The final splits.csv of notebook 01 was not found. Add Input → Your Work → Notebooks → '
                            'notebook 01, the version that made the 12,698 / 2,728 / 2,734 split.')
SPLITS_CSV = final_split[0]
splits = found[SPLITS_CSV]
classes = json.loads((SPLITS_CSV.parent / 'classes.json').read_text())
K = len(classes)
COLOR_DIR = find_color_dir(INPUT_ROOT)
train_counts = splits.loc[splits['split'] == 'train', 'label'].value_counts()
ORDER = [int(i) for i in train_counts.index]
CLASS_NAMES = [classes[i]['name'] for i in ORDER]
print('Split: ', SPLITS_CSV)
print('Images:', COLOR_DIR)

# Row A: notebook 02's final run
row_a_files = [f for f in find_files(INPUT_ROOT, '02_final_runs.json')]
if not row_a_files:
    raise FileNotFoundError('results/02_final_runs.json of notebook 02 was not found. Add Input → Your Work → '
                            'Notebooks → notebook 02, the version with FINAL = True.')
row_a = json.loads(row_a_files[0].read_text())
A_SETTINGS = row_a['settings']
same = {'img_size': IMG_SIZE, 'batch_size': BATCH_SIZE, 'max_epochs': 30, 'patience': PATIENCE,
        'start_lr': LR, 'optimizer': f'AdamW, weight decay {WEIGHT_DECAY}', 'split_fingerprint': SPLIT_FINGERPRINT,
        'normalisation': {'mean': MEAN, 'std': STD}}
different = {k: (A_SETTINGS.get(k), v) for k, v in same.items() if A_SETTINGS.get(k) != v}
assert not different, f'The settings differ from notebook 02 (notebook 02, here): {different}'
for r in row_a['runs']:
    r['row'] = 'A'
print(f'Row A: {len(row_a["runs"])} runs of notebook 02 ({row_a_files[0]}); settings checked: the same')

# The GAN images of notebook 03_gan_generate
synth_files = sorted(find_files(INPUT_ROOT, 'synthetic.csv'), key=lambda f: f.stat().st_mtime)
SYNTH_CSV = synth_files[-1] if synth_files else None
ROWS_READY = [r for r in ROWS if r in ('B', 'C') or SYNTH_CSV is not None]
if SYNTH_CSV:
    print('GAN images:', SYNTH_CSV)
elif any(r in ROWS for r in ('D', 'E')):
    print('No synthetic/synthetic.csv of notebook 03_gan_generate found: rows D and E wait. '
          'Add Input → Your Work → Notebooks → 03_gan_generate to train them.')

# Runs this notebook finished before (its last version, attached as input), and their saved models
state_files = find_files(INPUT_ROOT, f'{PREFIX}runs.json') + find_files(RESULTS_OUT, f'{PREFIX}runs.json')
previous = max((json.loads(f.read_text()) | {'_file': str(f)} for f in state_files),
               key=lambda s: len(s['runs']), default=None)
runs = previous['runs'] if previous else []
if previous:
    for run in runs:
        src = Path(previous['_file']).parent.parent / 'models' / f'{PREFIX}{run["row"]}_{run["model"]}.pt'
        dest = MODELS_OUT / src.name
        if src.exists() and not dest.exists():
            shutil.copy(src, dest)
    print(f'Continuing: {len(runs)} runs already done ({previous["_file"]})')
done = {(r['row'], r['model'], r['seed']) for r in runs}

## 2. The images

As in notebook 02: every image is read once, cropped to the centre square, resized to 224 × 224 (bilinear) and kept
in memory as 8-bit pixels. The GAN images are 256 × 256 like the PlantVillage photos and are read the same way. Per
class, the first `FILL_TARGET` − (real images) GAN images in `synthetic.csv` are used, so the class reaches 2,000.

In [ ]:
def load_image(path):
    """One image as IMG_SIZE x IMG_SIZE x 3 pixels (0-255): the centre square, resized bilinear, as in the web app."""
    with Image.open(path) as img:
        img = img.convert('RGB')
        side = min(img.size)
        left, top = (img.width - side) // 2, (img.height - side) // 2
        img = img.crop((left, top, left + side, top + side))
        return np.asarray(img.resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR))

def load_images(paths):
    x = np.empty((len(paths), IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    with ThreadPoolExecutor(2 * (os.cpu_count() or 2)) as pool:
        for i, pixels in enumerate(pool.map(load_image, paths)):
            x[i] = pixels
    return torch.from_numpy(x)

start = time.time()
data = {}
for name in ['train', 'val'] + ([] if SMOKE_TEST else ['test']):
    part = splits[splits['split'] == name]
    data[name] = (load_images([COLOR_DIR / p for p in part['path']]), torch.tensor(part['label'].to_numpy()))

synthetic = pd.DataFrame(columns=['path', 'label'])
if SYNTH_CSV is not None:
    synth_all = pd.read_csv(SYNTH_CSV)
    synthetic = pd.concat([
        synth_all[synth_all['label'] == i].head(max(0, FILL_TARGET - int(train_counts[i])))
        for i in range(K)], ignore_index=True)
synth_x = load_images([SYNTH_CSV.parent / p for p in synthetic['path']]) if len(synthetic) else \
    torch.empty((0, IMG_SIZE, IMG_SIZE, 3), dtype=torch.uint8)
synth_y = torch.tensor(synthetic['label'].to_numpy(dtype=np.int64))

for name, (x, y) in data.items():
    print(f'{name:5s} {len(x):6,} images  {x.nbytes / 1e9:.2f} GB')
real_y = data['train'][1]
REAL_N = len(real_y)
train_x = torch.cat([data.pop('train')[0], synth_x])   # real images first, then the GAN images
train_y = torch.cat([real_y, synth_y])
print(f'GAN   {len(synth_x):6,} images  {synth_x.nbytes / 1e9:.2f} GB')
print(f'Read in {time.time() - start:.0f} s')

counts = pd.DataFrame({'real': [int(train_counts[i]) for i in ORDER],
                       'GAN': [int((synthetic['label'] == i).sum()) for i in ORDER]}, index=CLASS_NAMES)
counts['real + GAN (rows D, E)'] = counts['real'] + counts['GAN']
counts['repeated (row C)'] = np.maximum(counts['real'], FILL_TARGET)
print('\nTraining images per class:\n')
print(counts.to_string())
short = counts[(counts['real + GAN (rows D, E)'] < FILL_TARGET) & (counts['GAN'] > 0)]
if SYNTH_CSV is not None and len(short):
    print(f'\nNote: {", ".join(short.index)} stay below {FILL_TARGET:,} in rows D and E '
          '(notebook 03_gan_generate kept fewer GAN images).')

## 3. The models and the training (the same as notebook 02)

The model code and the training loop are notebook 02's, unchanged. The only new parts are the order of the training
images in an epoch (rows C, D, E) and the augmentation (rows B, E):

- **Row C:** every epoch uses all real images, plus, for each class with fewer than 2,000, real images of that class
  drawn again (each one once before any is drawn twice) until it has 2,000. A new draw every epoch.
- **Rows D, E:** every epoch uses all real images and all the GAN images, shuffled together.
- **Augmentation (B, E):** every image, every time it is used, gets a random left-right and up-down flip (each with
  probability ½), a rotation of up to ±20°, a zoom of 1.0–1.2× (closer) and a brightness change of ±20%. The edges
  that a rotation uncovers are filled by mirroring. It runs on the GPU, on the whole batch at once.

The learning-rate schedule counts steps per epoch, so it runs over the same 30 epochs whatever the epoch size.

In [ ]:
class CustomCNN(nn.Module):
    """5 blocks of [3x3 conv + batch norm + ReLU] x 2 + 2x2 max pool; then average pool, dropout, linear layer."""

    def __init__(self, num_classes, widths=(32, 64, 128, 256, 512)):
        super().__init__()
        layers, c_in = [], 3
        for c in widths:
            for _ in range(2):
                layers += [nn.Conv2d(c_in, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True)]
                c_in = c
            layers.append(nn.MaxPool2d(2))
        self.features = nn.Sequential(*layers)
        self.classifier = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.2),
                                        nn.Linear(c_in, num_classes))

    def forward(self, x):
        return self.classifier(self.features(x))


def build_model(arch, pretrained):
    """The network with a new 10-class last layer, replaced the same way as in the web app (app/backend/model.py)."""
    if arch == 'custom_cnn':
        return CustomCNN(K)
    if arch == 'vgg16':
        net = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1 if pretrained else None)
        net.classifier[6] = nn.Linear(net.classifier[6].in_features, K)
    elif arch == 'resnet50':
        net = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
        net.fc = nn.Linear(net.fc.in_features, K)
    elif arch == 'mobilenet_v2':
        net = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V2 if pretrained else None)
        net.classifier[1] = nn.Linear(net.last_channel, K)
    else:
        raise ValueError(f'unknown model {arch}')
    return net

for arch in MODELS:   # downloads the ImageNet weights once
    try:
        build_model(arch, PRETRAINED)
    except Exception as e:
        raise RuntimeError(f'Could not build {arch}: {e}\nIf the ImageNet weights did not download: '
                           'Settings → Internet on, then run again.') from None

In [ ]:
MEAN_255 = torch.tensor(MEAN, device=DEVICE).view(1, 3, 1, 1) * 255
STD_255 = torch.tensor(STD, device=DEVICE).view(1, 3, 1, 1) * 255
CLASS_INDEX = [torch.nonzero(real_y == c).flatten() for c in range(K)]
AUGMENTED = {'B', 'E'}

def epoch_size(row):
    if row == 'B':
        return REAL_N
    if row == 'C':
        return REAL_N + sum(max(0, FILL_TARGET - len(idx)) for idx in CLASS_INDEX)
    return len(train_y)   # D, E

def epoch_order(row, gen):
    """The indices into train_x for one epoch, shuffled."""
    if row == 'B':
        return torch.randperm(REAL_N, generator=gen)
    if row == 'C':
        extra = []
        for idx in CLASS_INDEX:
            missing = FILL_TARGET - len(idx)
            if missing > 0:   # whole shuffled rounds of the class, so each image is drawn once before any twice
                rounds = [idx[torch.randperm(len(idx), generator=gen)] for _ in range(math.ceil(missing / len(idx)))]
                extra.append(torch.cat(rounds)[:missing])
        order = torch.cat([torch.arange(REAL_N), *extra])
        return order[torch.randperm(len(order), generator=gen)]
    return torch.randperm(len(train_y), generator=gen)   # D, E: real and GAN images together

def augment(x, gen):
    """Random flips, rotation, zoom and brightness for each image of a batch (n, 3, H, W) of 0-255 pixels."""
    n = x.shape[0]
    rand = lambda: torch.rand(n, device=x.device, generator=gen)
    flip_x = (1 - 2 * (rand() < 0.5).float()) if AUG['flip'] else torch.ones(n, device=x.device)
    flip_y = (1 - 2 * (rand() < 0.5).float()) if AUG['flip'] else torch.ones(n, device=x.device)
    angle = (2 * rand() - 1) * math.radians(AUG['rotate_degrees'])
    zoom = 1 + rand() * (AUG['zoom'] - 1)
    cos, sin, zero = torch.cos(angle) / zoom, torch.sin(angle) / zoom, torch.zeros(n, device=x.device)
    theta = torch.stack([torch.stack([cos * flip_x, -sin * flip_y, zero], 1),
                         torch.stack([sin * flip_x, cos * flip_y, zero], 1)], 1)   # output pixel -> input pixel
    grid = nn.functional.affine_grid(theta, list(x.shape), align_corners=False)
    x = nn.functional.grid_sample(x, grid, mode='bilinear', padding_mode='reflection', align_corners=False)
    brightness = 1 + (2 * rand() - 1) * AUG['brightness']
    return (x * brightness.view(n, 1, 1, 1)).clamp(0, 255)

def to_input(pixels, aug_gen=None):
    """8-bit images (n, H, W, 3) -> normalised float tensor (n, 3, H, W) on the GPU, augmented if aug_gen is given."""
    x = pixels.to(DEVICE, non_blocking=True).permute(0, 3, 1, 2).float()
    if aug_gen is not None:
        x = augment(x, aug_gen).contiguous(memory_format=torch.channels_last)
    return (x - MEAN_255) / STD_255

def autocast():
    return torch.autocast(DEVICE.type, dtype=torch.float16, enabled=AMP)

def scores(y_true, y_pred):
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=range(K), zero_division=0)
    return {'accuracy': float(np.mean(y_true == y_pred)), 'macro_precision': float(p.mean()),
            'macro_recall': float(r.mean()), 'macro_f1': float(f1.mean()),
            'precision': p.tolist(), 'recall': r.tolist(), 'f1': f1.tolist(),
            'confusion': confusion_matrix(y_true, y_pred, labels=range(K)).tolist()}

@torch.inference_mode()
def evaluate(net, split):
    x, y = data[split]
    net.eval()
    loss, preds = 0.0, []
    for i in range(0, len(x), EVAL_BATCH):
        with autocast():
            logits = net(to_input(x[i:i + EVAL_BATCH])).float()
        loss += nn.functional.cross_entropy(logits, y[i:i + EVAL_BATCH].to(DEVICE), reduction='sum').item()
        preds.append(logits.argmax(1).cpu())
    return loss / len(x), scores(y.numpy(), torch.cat(preds).numpy())

def train_epoch(net, opt, sched, scaler, order, aug_gen):
    net.train()
    total_loss = torch.zeros((), device=DEVICE)
    correct = torch.zeros((), device=DEVICE)
    for idx in order.split(BATCH_SIZE):
        xb, yb = to_input(train_x[idx], aug_gen), train_y[idx].to(DEVICE, non_blocking=True)
        with autocast():
            logits = net(xb)
            loss = nn.functional.cross_entropy(logits, yb)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        sched.step()
        total_loss += loss.detach().float() * len(idx)
        correct += (logits.argmax(1) == yb).sum()
    return total_loss.item() / len(order), correct.item() / len(order)

def better(a, b):
    """Is epoch a better than epoch b? Higher validation macro-F1; on equal macro-F1, lower validation loss."""
    return b is None or (a['val_macro_f1'], -a['val_loss']) > (b['val_macro_f1'], -b['val_loss'])

def train_run(arch, seed, row):
    """Train one model with one seed on one row's training images (notebook 02's train_run, plus the row)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    net = build_model(arch, PRETRAINED).to(DEVICE, memory_format=torch.channels_last)
    opt = torch.optim.AdamW(net.parameters(), lr=LR[arch], weight_decay=WEIGHT_DECAY)
    steps_per_epoch = math.ceil(epoch_size(row) / BATCH_SIZE)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS * steps_per_epoch)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=AMP)
    gen = torch.Generator().manual_seed(seed)
    aug_gen = torch.Generator(device=DEVICE).manual_seed(seed) if row in AUGMENTED else None
    history, best, best_weights = [], None, None
    for epoch in range(1, EPOCHS + 1):
        start = time.time()
        lr = opt.param_groups[0]['lr']
        train_loss, train_acc = train_epoch(net, opt, sched, scaler, epoch_order(row, gen), aug_gen)
        val_loss, val = evaluate(net, 'val')
        row_h = {'epoch': epoch, 'lr': lr, 'train_loss': train_loss, 'train_acc': train_acc, 'val_loss': val_loss,
                 'val_acc': val['accuracy'], 'val_macro_f1': val['macro_f1'], 'seconds': round(time.time() - start, 1)}
        history.append(row_h)
        is_best = better(row_h, best)
        if is_best:
            best = row_h
            best_weights = {k: v.detach().to('cpu', copy=True) for k, v in net.state_dict().items()}
        print(f'epoch {epoch:2d}/{EPOCHS}  train loss {train_loss:.4f} acc {100 * train_acc:6.2f}%  |  '
              f'val loss {val_loss:.4f} acc {100 * val["accuracy"]:6.2f}% macro-F1 {100 * val["macro_f1"]:6.2f}%  |  '
              f'{row_h["seconds"]:4.0f} s{"  best" if is_best else ""}')
        if epoch - best['epoch'] >= PATIENCE:
            print(f'Stopped early: no better validation macro-F1 for {PATIENCE} epochs')
            break
    net.load_state_dict(best_weights)
    return net, history, best

## 4. Train

Runs go row by row, model by model, seed by seed. Before each run the notebook estimates its longest possible time
(30 epochs, from the epoch times of notebook 02 or of this row's earlier runs); a run that might not finish within
`TIME_LIMIT_HOURS` is left for the next commit. After each run, `results/04_runs.json` is written and the best seed
of each row and model (by validation macro-F1, never by test) is saved as `models/04_<row>_<model>.pt`, in the web
app's format.

In [ ]:
def save_model(net, arch, row, run, path):
    metrics = {'validation_accuracy': run['val']['accuracy'], 'validation_macro_F1': run['val']['macro_f1']}
    if 'test' in run:
        metrics.update(test_accuracy=run['test']['accuracy'], test_macro_F1=run['test']['macro_f1'])
    torch.save({
        'arch': arch,
        'classes': classes,
        'state_dict': {k: v.detach().cpu().contiguous() for k, v in net.state_dict().items()},
        'img_size': IMG_SIZE,
        'mean': MEAN,
        'std': STD,
        'name': f'{NAMES[arch]} ({ROW_NAMES[row].lower()})',
        'metrics': metrics,
        'training_images': ROW_NAMES[row],
        'row': row,
        'seed': run['seed'],
        'best_epoch': run['best_epoch'],
        'run': 'smoke' if SMOKE_TEST else 'final',
    }, path)

settings = {
    'created_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'run': 'smoke' if SMOKE_TEST else 'final', 'seeds': SEEDS, 'scores_on': EVAL, 'models': MODELS,
    'rows': ROW_NAMES, 'row_A': f'notebook 02 final run ({row_a_files[0]})',
    'fill_target_rows_C_D_E': FILL_TARGET, 'augmentation_rows_B_E': AUG,
    'synthetic_csv': str(SYNTH_CSV) if SYNTH_CSV else None, 'synthetic_images': int(len(synthetic)),
    # the rest as in notebook 02 (checked above)
    **{k: A_SETTINGS[k] for k in ('img_size', 'batch_size', 'patience', 'early_stopping', 'optimizer', 'start_lr',
                                  'lr_schedule', 'loss', 'pretrained', 'normalisation', 'split_fingerprint')
       if k in A_SETTINGS},
    'max_epochs': EPOCHS, 'mixed_precision_fp16': AMP, 'split_csv': str(SPLITS_CSV),
    'device': torch.cuda.get_device_name() if DEVICE.type == 'cuda' else 'CPU', 'torch': torch.__version__,
}
STATE = RESULTS_OUT / f'{PREFIX}runs.json'
key = lambda run: (run['val']['macro_f1'], -run['val_loss'])

def best_run(row, arch):
    rs = [r for r in runs if r['row'] == row and r['model'] == arch]
    return max(rs, key=key) if rs else None

def epoch_seconds(row, arch):
    """Expected seconds per epoch: from this row's runs if there are any, otherwise from row A, scaled."""
    own = [h['seconds'] for r in runs if r['row'] == row and r['model'] == arch for h in r['history'][1:]]
    if own:
        return float(np.median(own))
    base = [h['seconds'] for r in row_a['runs'] if r['model'] == arch for h in r['history'][1:]]
    base = float(np.median(base)) if base else 60.0
    return base * epoch_size(row) / REAL_N * (1.15 if row in AUGMENTED else 1.0)

todo = [(row, arch, seed) for row in ROWS_READY for arch in MODELS for seed in SEEDS if (row, arch, seed) not in done]
print(f'{len(todo)} runs to do in rows {ROWS_READY}' + (f'; {len(done)} done before' if done else ''))
skipped = []
for row, arch, seed in todo:
    worst_minutes = (EPOCHS * epoch_seconds(row, arch) + 120) / 60   # all epochs, plus scoring and saving
    used_minutes = (time.time() - NOTEBOOK_START) / 60
    if used_minutes + worst_minutes > TIME_LIMIT_HOURS * 60:
        skipped.append((row, arch, seed))
        continue
    print(f'\n=== Row {row} ({ROW_NAMES[row]}), {NAMES[arch]}, seed {seed} — up to {worst_minutes:.0f} min ===')
    start = time.time()
    net, history, best = train_run(arch, seed, row)
    run = {'row': row, 'model': arch, 'seed': seed, 'best_epoch': best['epoch'], 'epochs_run': len(history),
           'epoch_size': epoch_size(row), 'minutes': round((time.time() - start) / 60, 1), 'history': history}
    run['val_loss'], run['val'] = evaluate(net, 'val')
    if not SMOKE_TEST:
        run['test_loss'], run['test'] = evaluate(net, 'test')
    previous_best = best_run(row, arch)
    runs.append(run)
    if previous_best is None or key(run) > key(previous_best):
        save_model(net, arch, row, run, MODELS_OUT / f'{PREFIX}{row}_{arch}.pt')
    STATE.write_text(json.dumps({'settings': settings, 'runs': runs}))
    print(f'Best epoch {run["best_epoch"]} of {run["epochs_run"]}: validation macro-F1 '
          f'{100 * run["val"]["macro_f1"]:.2f}%, {run["minutes"]} minutes')
    del net
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

STATE.write_text(json.dumps({'settings': settings, 'runs': runs}))
total = len(ROWS) * len(MODELS) * len(SEEDS)
finished = len({(r['row'], r['model'], r['seed']) for r in runs if r['row'] in ROWS})
if finished == total:
    print(f'\nAll {total} runs are done.')
else:
    print(f'\n{finished} of {total} runs done. Commit again with this notebook\'s latest version added as input to '
          f'train the other {total - finished}' + (' (rows D and E need the output of 03_gan_generate).'
                                                     if len(ROWS_READY) < len(ROWS) else '.'))

## 5. The comparison

Test scores (validation in a smoke test), the mean ± standard deviation over the 3 seeds. Row A is notebook 02's final
run. With 2,734 test images one image is 0.04 percentage points of accuracy, and the models make only 6–37 mistakes
per run, so differences smaller than about one standard deviation are not real differences.

In [ ]:
def pct(values):
    v = 100 * np.asarray(values, dtype=float)
    return f'{v.mean():.2f} ± {v.std(ddof=1):.2f}' if len(v) > 1 else f'{v[0]:.2f}'

all_runs = ([] if SMOKE_TEST else row_a['runs']) + runs
group = lambda row, arch: [r for r in all_runs if r['row'] == row and r['model'] == arch and EVAL in r]
ROWS_SHOWN = [r for r in ALL_ROWS if any(group(r, a) for a in MODELS)]

results = pd.DataFrame([{
    'Row': row, 'Training images': ROW_NAMES[row], 'Model': NAMES[arch], 'Seeds': len(rs),
    'Accuracy %': pct([r[EVAL]['accuracy'] for r in rs]),
    'Macro precision %': pct([r[EVAL]['macro_precision'] for r in rs]),
    'Macro recall %': pct([r[EVAL]['macro_recall'] for r in rs]),
    'Macro-F1 %': pct([r[EVAL]['macro_f1'] for r in rs]),
    'Validation macro-F1 %': pct([r['val']['macro_f1'] for r in rs]),
    'Best epoch': ', '.join(str(r['best_epoch']) for r in rs),
    'Images per epoch': rs[0].get('epoch_size', REAL_N),
    'Training min / seed': round(float(np.mean([r['minutes'] for r in rs])), 1),
} for row in ROWS_SHOWN for arch in MODELS for rs in [group(row, arch)] if rs])
results.to_csv(RESULTS_OUT / f'{PREFIX}results.csv', index=False)

pivot = results.pivot(index='Row', columns='Model', values='Macro-F1 %').reindex(columns=[NAMES[a] for a in MODELS])
pivot.index = [f'{r}  {ROW_NAMES[r]}' for r in pivot.index]
print(f'Macro-F1 % on the {EVAL} split (mean ± sd over the seeds):\n')
print(pivot.fillna('').to_string())
print(f'\nAccuracy % on the {EVAL} split:\n')
acc = results.pivot(index='Row', columns='Model', values='Accuracy %').reindex(columns=[NAMES[a] for a in MODELS])
acc.index = pivot.index
print(acc.fillna('').to_string())

pd.DataFrame([{'row': r['row'], 'model': NAMES[r['model']], 'seed': r['seed'], 'best_epoch': r['best_epoch'],
               'epochs_run': r['epochs_run'], 'minutes': r['minutes'], 'val_macro_f1': r['val']['macro_f1'],
               **{f'{EVAL}_{m}': r[EVAL][m] for m in ('accuracy', 'macro_precision', 'macro_recall', 'macro_f1')}}
              for r in all_runs if EVAL in r]).to_csv(RESULTS_OUT / f'{PREFIX}runs.csv', index=False)

per_class = pd.DataFrame([{'row': row, 'model': NAMES[arch], 'class': classes[i]['name'],
                           'train images': int(train_counts[i]),
                           **{f'{m} %': pct([r[EVAL][m][i] for r in rs]) for m in ('precision', 'recall', 'f1')}}
                          for row in ROWS_SHOWN for arch in MODELS for rs in [group(row, arch)] if rs
                          for i in ORDER])
per_class.to_csv(RESULTS_OUT / f'{PREFIX}per_class.csv', index=False)

# recall per class and row, mean over the models and seeds
recall = pd.DataFrame({row: [100 * np.mean([r[EVAL]['recall'][i] for a in MODELS for r in group(row, a)])
                             for i in ORDER] for row in ROWS_SHOWN}, index=CLASS_NAMES)
print(f'\nRecall per class % ({EVAL} split), mean over the models and seeds:\n')
print(recall.round(2).to_string())

In [ ]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(3.4 * len(MODELS), 4.2), sharey=True, squeeze=False)
axes = axes[0]
for ax, arch in zip(axes, MODELS):
    for x, row in enumerate(ROWS_SHOWN):
        rs = group(row, arch)
        if not rs:
            continue
        v = 100 * np.array([r[EVAL]['macro_f1'] for r in rs])
        ax.plot(np.full(len(v), x) + np.linspace(-0.12, 0.12, len(v)), v, 'o', color=BLUE, alpha=0.45, markersize=5,
                markeredgewidth=0)
        ax.errorbar(x + 0.28, v.mean(), yerr=v.std(ddof=1) if len(v) > 1 else 0, fmt='o', color=INK, markersize=7,
                    markeredgecolor=SURFACE, markeredgewidth=1.5, elinewidth=1.5, capsize=3)
    ax.set_xticks(range(len(ROWS_SHOWN)), ROWS_SHOWN)
    ax.set_xlim(-0.5, len(ROWS_SHOWN) - 0.3)
    ax.set_title(NAMES[arch])
    ax.yaxis.grid(True)
    ax.set_axisbelow(True)
    ax.set_xlabel('Row')
axes[0].set_ylabel(f'Macro-F1 % ({EVAL} split)')
fig.tight_layout(rect=(0, 0.06, 1, 0.88))
fig.legend(handles=[plt.Line2D([], [], color=BLUE, marker='o', linestyle='', alpha=0.45, label='one seed'),
                    plt.Line2D([], [], color=INK, marker='o', linestyle='', label='mean ± sd')],
           loc='lower center', ncol=2, frameon=False, bbox_to_anchor=(0.5, 0))
fig.suptitle(f'Macro-F1 by training images, {EVAL} split', x=0.01, y=0.995, ha='left', fontweight='bold', fontsize=13)
fig.text(0.01, 0.925, '   '.join(f'{r}: {ROW_NAMES[r]}' for r in ROWS_SHOWN), color=INK_2, fontsize=9)
fig.savefig(RESULTS_OUT / f'{PREFIX}macro_f1.png')
plt.show()

if 'A' in ROWS_SHOWN and len(ROWS_SHOWN) > 1:
    change = recall[[r for r in ROWS_SHOWN if r != 'A']].sub(recall['A'], axis=0)
    limit = max(0.5, float(np.abs(change.to_numpy()).max()))
    fig, ax = plt.subplots(figsize=(2.8 + 1.3 * change.shape[1], 6))
    ax.imshow(change.to_numpy(), cmap=DIVERGING, norm=TwoSlopeNorm(0, -limit, limit), aspect='auto')
    for i in range(change.shape[0]):
        for j in range(change.shape[1]):
            v = change.iat[i, j]
            ax.text(j, i, f'{v:+.1f}', ha='center', va='center', fontsize=10,
                    color='white' if abs(v) > 0.6 * limit else INK)
    ax.set_xticks(range(change.shape[1]), change.columns)
    ax.set_yticks(range(K), [f'{n}  ({train_counts[i]:,})' for n, i in zip(CLASS_NAMES, ORDER)])
    ax.set_xticks(np.arange(change.shape[1] + 1) - 0.5, minor=True)
    ax.set_yticks(np.arange(K + 1) - 0.5, minor=True)
    ax.grid(which='minor', color=SURFACE, linewidth=2)
    ax.grid(which='major', visible=False)
    ax.tick_params(which='both', length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_title(f'Recall change against row A (percentage points), {EVAL} split', pad=24)
    ax.text(0, 1.015, f'Blue: better than real only, red: worse. Mean of {len(MODELS)} models × {len(SEEDS)} seeds. '
            'Brackets: real training images.', transform=ax.transAxes, color=INK_2, fontsize=9)
    fig.savefig(RESULTS_OUT / f'{PREFIX}recall_change.png')
    plt.show()

In [ ]:
def confusion_sum(row, arch):
    """The confusion matrices of all seeds added up (rows: true class), in the chart order of the classes."""
    return np.sum([np.array(r[EVAL]['confusion']) for r in group(row, arch)], axis=0)[np.ix_(ORDER, ORDER)]

pair = [r for r in ('A', 'D') if r in ROWS_SHOWN]
if len(pair) == 2:
    for arch in MODELS:
        if not (group('A', arch) and group('D', arch)):
            continue
        fig, axes = plt.subplots(1, 2, figsize=(14, 6.8), gridspec_kw={'wspace': 0.04})
        axes[0].set_anchor('E')   # the two square matrices close together
        axes[1].set_anchor('W')
        for ax, row in zip(axes, pair):
            cm = confusion_sum(row, arch)
            share = cm / np.maximum(cm.sum(1, keepdims=True), 1)
            ax.imshow(share, cmap=BLUES, norm=PowerNorm(0.5, vmin=0, vmax=1))
            for i in range(K):
                for j in range(K):
                    if cm[i, j]:
                        ax.text(j, i, f'{cm[i, j]:,}', ha='center', va='center', fontsize=8,
                                color='white' if share[i, j] > 0.3 else INK)
            ax.set_xticks(range(K), CLASS_NAMES, rotation=40, ha='right', rotation_mode='anchor', fontsize=9)
            ax.set_yticks(range(K), CLASS_NAMES if row == 'A' else [''] * K, fontsize=9)
            ax.set_xticks(np.arange(K + 1) - 0.5, minor=True)
            ax.set_yticks(np.arange(K + 1) - 0.5, minor=True)
            ax.grid(which='minor', color=SURFACE, linewidth=2)
            ax.grid(which='major', visible=False)
            ax.tick_params(which='both', length=0)
            for spine in ax.spines.values():
                spine.set_visible(False)
            errors = int(cm.sum() - np.trace(cm))
            ax.set_title(f'{row}: {ROW_NAMES[row]}  ({errors} mistakes)', fontsize=12)
            ax.set_xlabel('Predicted class')
        axes[0].set_ylabel('True class')
        fig.suptitle(f'{NAMES[arch]}: confusion matrices, {EVAL} split, the {len(group("A", arch))} seeds added up',
                     x=0.01, ha='left', fontweight='bold', fontsize=13)
        fig.tight_layout()
        fig.savefig(RESULTS_OUT / f'{PREFIX}confusion_A_vs_D_{arch}.png')
        plt.show()

## 6. Save, and the model for the web app

In [ ]:
complete = {(row, arch): len(group(row, arch)) == len(SEEDS) for row in ROWS_SHOWN for arch in MODELS}
print('For app/frontend/src/data/results.js (notebook 05 writes this file):\n')
for row in ROWS_SHOWN:
    for arch in MODELS:
        rs = group(row, arch)
        if rs and complete[(row, arch)]:
            print(f"  {{ model: '{NAMES[arch]}', images: '{ROW_NAMES[row]}', "
                  f"accuracy: {np.mean([r[EVAL]['accuracy'] for r in rs]):.4f}, "
                  f"macroF1: {np.mean([r[EVAL]['macro_f1'] for r in rs]):.4f} }},")

print('\nMean validation macro-F1 % (the app model is chosen on these, never on test):\n')
val_table = pd.DataFrame({NAMES[a]: {row: 100 * np.mean([r['val']['macro_f1'] for r in group(row, a)])
                                     for row in ROWS_SHOWN if group(row, a)} for a in MODELS})
print(val_table.round(2).to_string())
if 'MobileNetV2' in val_table:
    best_row = val_table['MobileNetV2'].idxmax()
    print(f'\nMobileNetV2 is best in row {best_row} ({ROW_NAMES[best_row]}): notebook 05 uses this one for the app '
          '(unless APP_ARCH or APP_ROW is set there).')

STATE.write_text(json.dumps({'settings': settings, 'runs': runs}, indent=1))
for f in sorted(OUT_DIR.rglob('*')):
    if f.is_file() and f.parent.name in ('results', 'models') and f.name.startswith(PREFIX):
        print(f'{f.relative_to(OUT_DIR)}  ({f.stat().st_size / 1e6:,.1f} MB)')
print(f'\nThis run: {(time.time() - NOTEBOOK_START) / 3600:.1f} hours')

## Checklist

1. **All runs done?** The last line of section 4 says so. Until then the tables show the rows finished so far.
2. **Training curves:** `04_runs.json` holds every epoch. Augmented rows (B, E) usually need more epochs and reach a
   slightly higher training loss; that is expected.
3. **The macro-F1 table and chart:** is D above A, B and C, and by more than the spread of the seeds? If not, that is
   the result, and it is reported as it is.
4. **Recall change** (`04_recall_change.png`): where the GAN helps. The plan expects any gain mostly in the small
   classes (Mosaic virus, Leaf mold, Early blight). Compare D with C: a gain over C is what the GAN adds beyond
   balancing.
5. **Confusion matrices A vs D:** which mix-ups (for example Early blight ↔ Late blight / Target spot) got fewer.

**Next:** download `results/` into the project folder as `results-04/`. The `models/` files can stay on Kaggle;
notebook 05 picks the app model from them and writes everything the web app needs.